## 任务0：环境、路径和个人信息

从项目根目录启动Jupyter，按任务顺序运行。断言用于核对数据口径和成果文件，不应删除。

# 第10天：分类模型比较、选择与业务应用

逻辑回归、决策树和随机森林使用完全相同的训练集、测试集与预处理流程，再用准确率、流失召回率和混淆矩阵进行公平比较。

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_PATH = PROJECT_ROOT / 'data' / 'ecommerce_customer_cleaned.csv'
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('项目目录：', PROJECT_ROOT.resolve())
print('数据文件：', DATA_PATH.resolve())

项目目录： /mnt/data/work_240124/stu/muc-commerce-2-24012412-main/day10_model_comparison_student
数据文件： /mnt/data/work_240124/stu/muc-commerce-2-24012412-main/day10_model_comparison_student/data/ecommerce_customer_cleaned.csv


In [2]:
STUDENT_NAME = "24012412"
STUDENT_ID = "24012412"
CLASS_NAME = "信计二班"
assert STUDENT_NAME.strip() and STUDENT_ID == "24012412" and CLASS_NAME.strip(), "个人信息不能为空"

## 任务1：沿用第9天的数据口径

一行表示一名用户，`Churn`是标签，`CustomerID`只在输出预测名单时用于回查，不能进入模型特征。

In [3]:
df = pd.read_csv(DATA_PATH)
print('数据形状：', df.shape)
print('总体流失率：', f"{df['Churn'].mean():.2%}")
assert df.shape == (5630, 22)
assert df['CustomerID'].is_unique
assert set(df['Churn'].unique()) == {0, 1}
assert df.isna().sum().sum() == 0

数据形状： (5630, 22)
总体流失率： 16.84%


In [4]:
TARGET = "Churn"
ID_COL = "CustomerID"
feature_columns = [c for c in df.columns if c not in {TARGET, ID_COL}]
X = df.loc[:, feature_columns].copy()
y = df[TARGET].astype(int).copy()
customer_ids = df[ID_COL].copy()
assert TARGET not in X.columns and ID_COL not in X.columns
print("特征数：", X.shape[1], "标签流失人数：", int(y.sum()))

特征数： 20 标签流失人数： 948


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
test_customer_ids = customer_ids.loc[X_test.index]
print("训练集：", X_train.shape, f"流失率={y_train.mean():.2%}")
print("测试集：", X_test.shape, f"流失率={y_test.mean():.2%}")
assert len(X_train) == 4504 and len(X_test) == 1126
assert abs(y_train.mean() - y_test.mean()) < 0.001

训练集： (4504, 20) 流失率=16.83%
测试集： (1126, 20) 流失率=16.87%


## 任务2：训练逻辑回归——综合多个证据形成判断

逻辑回归会输出流失概率，再根据阈值给出0或1。`class_weight='balanced'`由教师预设，用于提醒模型不要忽略人数较少的流失用户。

In [6]:
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

def build_preprocessor():
    numeric_branch = Pipeline([
        ("fill_median", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_branch = Pipeline([
        ("fill_mode", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric_branch, numeric_features),
        ("cat", categorical_branch, categorical_features),
    ])

def build_pipeline(model):
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", model),
    ])

fitted_models = {}
predictions = {}
probabilities = {}

In [7]:
logistic_pipeline = build_pipeline(LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
))
logistic_pipeline.fit(X_train, y_train)
fitted_models["logistic_regression"] = logistic_pipeline
predictions["logistic_regression"] = logistic_pipeline.predict(X_test)
probabilities["logistic_regression"] = logistic_pipeline.predict_proba(X_test)[:, 1]
print("逻辑回归训练完成；预测流失人数：", int(predictions["logistic_regression"].sum()))

逻辑回归训练完成；预测流失人数： 357


## 任务3：训练决策树——连续提出若干判断问题

`max_depth=5`限制树不要无限追问；`min_samples_leaf=20`避免只根据极少数用户形成叶子。

In [8]:
tree_pipeline = build_pipeline(DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=RANDOM_STATE,
))
tree_pipeline.fit(X_train, y_train)
fitted_models["decision_tree"] = tree_pipeline
predictions["decision_tree"] = tree_pipeline.predict(X_test)
probabilities["decision_tree"] = tree_pipeline.predict_proba(X_test)[:, 1]
print("决策树训练完成；预测流失人数：", int(predictions["decision_tree"].sum()))

决策树训练完成；预测流失人数： 385


## 任务4：训练随机森林——让多棵树共同投票

教师固定使用100棵树。今天只理解“多棵树投票通常比一棵树更稳定”，不开展参数搜索。

In [9]:
forest_pipeline = build_pipeline(RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
))
forest_pipeline.fit(X_train, y_train)
fitted_models["random_forest"] = forest_pipeline
predictions["random_forest"] = forest_pipeline.predict(X_test)
probabilities["random_forest"] = forest_pipeline.predict_proba(X_test)[:, 1]
print("随机森林训练完成；预测流失人数：", int(predictions["random_forest"].sum()))

随机森林训练完成；预测流失人数： 271


## 任务5：用同一张成绩单比较模型

最低参照线、三个正式模型必须使用同一测试集。混淆矩阵中的四个数分别是TN、FP、FN、TP。

In [10]:
baseline_pipeline = build_pipeline(DummyClassifier(strategy="prior", random_state=RANDOM_STATE))
baseline_pipeline.fit(X_train, y_train)
fitted_models["baseline"] = baseline_pipeline
predictions["baseline"] = baseline_pipeline.predict(X_test)
probabilities["baseline"] = baseline_pipeline.predict_proba(X_test)[:, 1]

def evaluate_model(model_name):
    pred = predictions[model_name]
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "churn_recall": recall_score(y_test, pred, zero_division=0),
        "predicted_churn_count": int(pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [11]:
model_order = ["baseline", "logistic_regression", "decision_tree", "random_forest"]
model_comparison = pd.DataFrame([evaluate_model(name) for name in model_order])
model_comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")
display(model_comparison.style.format({
    "accuracy": "{:.2%}",
    "precision": "{:.2%}",
    "churn_recall": "{:.2%}",
}))

,model,accuracy,precision,churn_recall,predicted_churn_count,tn,fp,fn,tp
0,baseline,83.13%,0.00%,0.00%,0,936,0,190,0
1,logistic_regression,79.84%,44.82%,84.21%,357,739,197,30,160
2,decision_tree,77.71%,42.08%,85.26%,385,713,223,28,162
3,random_forest,88.01%,60.15%,85.79%,271,828,108,27,163


In [12]:
confusion_summary = model_comparison.loc[:, ["model", "tn", "fp", "fn", "tp"]].copy()
confusion_summary["total"] = confusion_summary[["tn", "fp", "fn", "tp"]].sum(axis=1)
confusion_summary.to_csv(OUTPUT_DIR / "confusion_matrix_summary.csv", index=False, encoding="utf-8-sig")
assert confusion_summary["total"].eq(len(y_test)).all()
display(confusion_summary)

,model,tn,fp,fn,tp,total
0,baseline,936,0,190,0,1126
1,logistic_regression,739,197,30,160,1126
2,decision_tree,713,223,28,162,1126
3,random_forest,828,108,27,163,1126


## 任务6：选择最终模型并说明理由

不能只写“准确率最高”。至少同时比较流失召回率、漏掉人数FN和误报人数FP。

In [13]:
SELECTED_MODEL_NAME = "random_forest"
allowed_models = {"logistic_regression", "decision_tree", "random_forest"}
assert SELECTED_MODEL_NAME in allowed_models, "最终模型必须从三个正式模型中选择"
selected_pipeline = fitted_models[SELECTED_MODEL_NAME]
selected_prediction = predictions[SELECTED_MODEL_NAME]
selected_probability = probabilities[SELECTED_MODEL_NAME]
print("最终模型：", SELECTED_MODEL_NAME)

最终模型： random_forest


In [14]:
selection_note = "随机森林的准确率约为88.0%，流失召回率约为85.8%，在三个正式模型中同时保持较高识别能力和较少误报。它只漏掉27名流失用户，FN与决策树接近，但FP明显更少；相较逻辑回归也减少了误报。因此本次选择随机森林，用于形成优先联系名单。"
assert 80 <= len(selection_note) <= 180, "模型选择说明应为80～180字"
(OUTPUT_DIR / "model_selection_note.txt").write_text(selection_note, encoding="utf-8")
print(selection_note)

随机森林的准确率约为88.0%，流失召回率约为85.8%，在三个正式模型中同时保持较高识别能力和较少误报。它只漏掉27名流失用户，FN与决策树接近，但FP明显更少；相较逻辑回归也减少了误报。因此本次选择随机森林，用于形成优先联系名单。


## 任务7：输出用户预测与高风险名单

流失概率只表示模型对风险程度的估计，数值越高越值得优先核查，但不能描述成用户一定会流失。

In [15]:
customer_predictions = pd.DataFrame({
    "CustomerID": test_customer_ids.to_numpy(),
    "actual_churn": y_test.to_numpy(),
    "predicted_churn": selected_prediction.astype(int),
    "churn_probability": selected_probability,
})
customer_predictions["prediction_correct"] = customer_predictions["actual_churn"].eq(
    customer_predictions["predicted_churn"]
)
customer_predictions.to_csv(
    OUTPUT_DIR / "customer_churn_predictions.csv", index=False, encoding="utf-8-sig"
)
display(customer_predictions.head())
assert len(customer_predictions) == 1126
assert customer_predictions["CustomerID"].is_unique
assert customer_predictions["churn_probability"].between(0, 1).all()

,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,54007,0,0,0.132813,True
1,51970,0,0,0.091591,True
2,54236,0,0,0.152303,True
3,50106,0,0,0.064959,True
4,52296,0,0,0.349675,True


In [16]:
high_risk_customers = customer_predictions.loc[
    customer_predictions["predicted_churn"].eq(1)
].sort_values("churn_probability", ascending=False).reset_index(drop=True)
high_risk_customers.to_csv(
    OUTPUT_DIR / "high_risk_customers.csv", index=False, encoding="utf-8-sig"
)
print("进入优先关注名单的人数：", len(high_risk_customers))
display(high_risk_customers.head(10))

进入优先关注名单的人数： 271


,CustomerID,actual_churn,predicted_churn,churn_probability,prediction_correct
0,54618,1,1,0.961114,True
1,54288,1,1,0.955265,True
2,54023,1,1,0.942563,True
3,50936,1,1,0.938924,True
4,53751,1,1,0.934535,True
5,50339,1,1,0.932899,True
6,54870,1,1,0.931667,True
7,54990,1,1,0.926240,True
8,51215,1,1,0.917427,True
9,54263,1,1,0.914952,True


In [17]:
selected_preprocessor = selected_pipeline.named_steps["preprocessor"]
selected_model = selected_pipeline.named_steps["model"]
feature_names = selected_preprocessor.get_feature_names_out()

if hasattr(selected_model, "feature_importances_"):
    importance_values = selected_model.feature_importances_
elif hasattr(selected_model, "coef_"):
    importance_values = np.abs(selected_model.coef_[0])
else:
    importance_values = np.zeros(len(feature_names))

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance_values,
}).sort_values("importance", ascending=False).reset_index(drop=True)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False, encoding="utf-8-sig")
display(feature_importance.head(10))

,feature,importance
0,num__Tenure,0.274057
1,cat__TenureGroup_0-6个月,0.128479
2,num__Complain,0.081726
3,num__CashbackAmount,0.067729
4,num__DaySinceLastOrder,0.048221
5,num__NumberOfAddress,0.038737
6,num__SatisfactionScore,0.031896
7,num__WarehouseToHome,0.029742
8,cat__PreferedOrderCat_Mobile Phone,0.028944
9,cat__TenureGroup_13-24个月,0.026077


## 任务8：保存并重新加载模型

保存的是完整流水线，因此原始用户表可以按同样规则完成预处理和预测。

In [18]:
MODEL_PATH = OUTPUT_DIR / "selected_model.joblib"
joblib.dump(selected_pipeline, MODEL_PATH)
reloaded_pipeline = joblib.load(MODEL_PATH)
reloaded_prediction = reloaded_pipeline.predict(X_test)
assert np.array_equal(reloaded_prediction, selected_prediction)

metadata = {
    "student_id": STUDENT_ID,
    "selected_model": SELECTED_MODEL_NAME,
    "random_state": RANDOM_STATE,
    "test_rows": len(X_test),
    "feature_columns": X.columns.tolist(),
}
(OUTPUT_DIR / "model_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("模型已保存并通过重新加载检查：", MODEL_PATH)

模型已保存并通过重新加载检查： /mnt/data/work_240124/stu/muc-commerce-2-24012412-main/day10_model_comparison_student/output/selected_model.joblib


## 任务9：完成学习复盘

请解释为什么最低参照线不可用、三个模型为什么必须公平比较，以及最终模型如何用于业务筛查。

In [19]:
reflection = "最低参照线虽然准确率达到83%左右，但它把测试用户全部判断为不流失，流失召回率为0，因此不能解决客户挽留问题。逻辑回归、决策树和随机森林只有使用同一次分层划分、同一套预处理和同一个测试集，准确率、召回率以及FP、FN才有可比性。本次选择随机森林，是因为它在保持较高召回率的同时明显减少误报。输出的概率应作为筛查依据，业务人员可优先核查高风险名单，再结合投诉记录和沟通结果决定是否采取优惠或回访措施，不能把概率当作确定事实。"
assert 150 <= len(reflection) <= 250, "学习复盘应为150～250字"
(OUTPUT_DIR / "reflection.txt").write_text(reflection, encoding="utf-8")
print(reflection)

最低参照线虽然准确率达到83%左右，但它把测试用户全部判断为不流失，流失召回率为0，因此不能解决客户挽留问题。逻辑回归、决策树和随机森林只有使用同一次分层划分、同一套预处理和同一个测试集，准确率、召回率以及FP、FN才有可比性。本次选择随机森林，是因为它在保持较高召回率的同时明显减少误报。输出的概率应作为筛查依据，业务人员可优先核查高风险名单，再结合投诉记录和沟通结果决定是否采取优惠或回访措施，不能把概率当作确定事实。


## 提交检查

In [20]:
required = {
    "model_comparison.csv",
    "confusion_matrix_summary.csv",
    "customer_churn_predictions.csv",
    "high_risk_customers.csv",
    "feature_importance.csv",
    "selected_model.joblib",
    "model_metadata.json",
    "model_selection_note.txt",
    "reflection.txt",
}
actual = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
missing = required - actual
print("成果文件：", sorted(actual))
assert not missing, f"缺少成果文件：{sorted(missing)}"
print("第10天Notebook检查通过")

成果文件： ['confusion_matrix_summary.csv', 'customer_churn_predictions.csv', 'feature_importance.csv', 'high_risk_customers.csv', 'model_comparison.csv', 'model_metadata.json', 'model_selection_note.txt', 'reflection.txt', 'selected_model.joblib']
第10天Notebook检查通过
